# Flat look-alike dataset — construction

Builds **one row per company** for the look-alike propensity model: *"which non-customers most
resemble existing Lloyds customers?"*

## Why this design

The panel (one row per company × cutoff, forward 13-week label) has 0.02% positives, which is why
Precision@K sat at zero. This table asks an easier, **business-aligned** question instead — §9 of the
brief already defines the product as high-scoring **non-customers** (look-alike prospects):

| design | positives | prevalence |
|---|---|---|
| panel, forward 13wk | 426 | 0.02% |
| **flat, `label_mode="ever"`** | **~8,400** | **~2.7%** |
| flat, `label_mode="since"` (2015+) | ~2,800 | ~1.0% |

No GDELT here: the ablation showed `tone_z`/`vol_z` add nothing (gap ≈ 0 at every K, both
coefficients' clustered CIs cross zero), so the media pillar is out of the model and this table has
no reason to live in the GDELT notebook.

## The one trap — read before choosing `LABEL_MODE`

With `"ever"`, **age partly measures exposure time, not propensity**: a 30-year-old firm has had 30
years of chances to take a Lloyds charge, a 3-year-old has had 3. Even with identical propensity,
P(ever) would be 45% vs 6%. Section 6 prints the diagnostic — under `"ever"` the oldest firms look
like the *best* segment, but under a fixed recent window they are the *worst*. Prefer
`LABEL_MODE="since"` unless you specifically want the profiling framing, and never read the `age`
coefficient causally under `"ever"`.

## 1 · Config — the knobs you manage

In [13]:
# --- portable paths: resolve the project root from ANY working directory ---
import sys
from pathlib import Path
_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "API").is_dir())
sys.path.insert(0, str(_ROOT))
from paths import SME_CSV, CHARGES_CSV, FLAT_CSV

import numpy as np
import pandas as pd

# ============================ SETTINGS ============================
# Snapshot date. EVERY feature is measured as-of here, and charge history is cut
# here, so no feature can see the future. Keep it <= the last date in
# charges_history.csv or you silently lose recent charges from the features.
ASOF = pd.Timestamp("2026-01-01")

# "ever"  -> label = 1 if the firm has ANY Lloyds charge, ever      (~8.4k positives, contaminated)
# "since" -> label = 1 if its FIRST Lloyds charge is >= LABEL_SINCE (~2.8k, equal exposure, cleaner)
#            firms that converted BEFORE LABEL_SINCE are dropped: they were already customers.
LABEL_MODE  = "ever"
LABEL_SINCE = pd.Timestamp("2015-01-01")

OUT_CSV = FLAT_CSV
# ==================================================================
print(f"ASOF={ASOF.date()}  LABEL_MODE={LABEL_MODE}"
      + (f"  LABEL_SINCE={LABEL_SINCE.date()}" if LABEL_MODE == "since" else ""))

ASOF=2026-01-01  LABEL_MODE=ever


## 2 · Load the companies and their charge history

`charges_history.csv` is the label source (`is_lloyds`, set by the curated `LLOYDS_PATTERNS` regex in
Stage 3 of the CH pipeline). Non-Lloyds charges are safe as features — only Lloyds ones are the label.

In [14]:
companies = pd.read_csv(SME_CSV, dtype=str, low_memory=False)
charges   = pd.read_csv(CHARGES_CSV, dtype=str)

companies["born"] = pd.to_datetime(companies["date_of_creation"], errors="coerce")
companies = companies.dropna(subset=["born"]).reset_index(drop=True)

charges["created_on"] = pd.to_datetime(charges["created_on"], errors="coerce")
charges["is_lloyds"]  = charges["is_lloyds"].astype(str).str.lower().eq("true")
charges = charges.dropna(subset=["created_on"])

# THE POINT-IN-TIME CUT. Without it, features read charges filed after ASOF and
# yrs_since_last_charge goes NEGATIVE (3,768 such rows in the previous flat.csv).
charges_asof = charges[charges["created_on"] <= ASOF]

first_lloyds = charges_asof[charges_asof["is_lloyds"]].groupby("com_num")["created_on"].min()
nonlloyds    = charges_asof[~charges_asof["is_lloyds"]]

companies["first_lloyds"] = companies["com_num"].map(first_lloyds)

print(f"companies            : {len(companies):,}")
print(f"charge rows (all)    : {len(charges):,}")
print(f"charge rows <= ASOF  : {len(charges_asof):,}   "
      f"({len(charges) - len(charges_asof):,} dropped as future)")
print(f"ever-Lloyds by ASOF  : {companies['first_lloyds'].notna().sum():,}")

companies            : 311,994
charge rows (all)    : 209,607
charge rows <= ASOF  : 201,788   (7,819 dropped as future)
ever-Lloyds by ASOF  : 10,122


## 3 · Define the label

`"ever"` keeps everyone and asks *"is this firm a customer?"*. `"since"` restricts to a fixed recent
window so every surviving firm had the **same opportunity** to convert, and drops firms that were
already customers when the window opened.

In [15]:
d = companies.copy()

if LABEL_MODE == "ever":
    d["label"] = d["first_lloyds"].notna().astype(int)

elif LABEL_MODE == "since":
    already = d["first_lloyds"].notna() & (d["first_lloyds"] < LABEL_SINCE)
    born_in_time = d["born"] <= LABEL_SINCE          # equal exposure across the window
    d = d[~already & born_in_time].copy()
    d["label"] = ((d["first_lloyds"] >= LABEL_SINCE) & (d["first_lloyds"] <= ASOF)).astype(int)

else:
    raise ValueError("LABEL_MODE must be 'ever' or 'since'")

d = d.reset_index(drop=True)
print(f"rows      : {len(d):,}")
print(f"positives : {int(d['label'].sum()):,}  ({d['label'].mean():.2%})")

rows      : 311,994
positives : 10,122  (3.24%)


## 4 · Features, measured as-of `ASOF`

All charge features come from `nonlloyds` (already cut at `ASOF`), so none of them can see the
future. `sector`, `region` and `account_type` are still **current snapshots** — Companies House does
not serve history for them cheaply. They are near-static, but note it as a caveat.

`age_years` is included for **diagnostics**; think hard before modelling on it under
`LABEL_MODE="ever"` (see section 6).

In [16]:
g = nonlloyds.groupby("com_num")

d["nonlloyds_charges"]    = d["com_num"].map(g.size()).fillna(0).astype(int)
d["has_nonlloyds_charge"] = (d["nonlloyds_charges"] > 0).astype(int)
d["n_distinct_lenders"]   = d["com_num"].map(g["persons_entitled"].nunique()).fillna(0).astype(int)

# 50 = "never borrowed" sentinel; clip keeps the never-borrowed and long-ago cases together
_last = d["com_num"].map(g["created_on"].max())
d["yrs_since_nonlloyds_chg"] = ((ASOF - _last).dt.days / 365.25).fillna(50).clip(0, 50)

d["age_years"] = (ASOF - d["born"]).dt.days / 365.25
d["accounts_overdue"] = d["accounts_overdue"].map(
    {"True": 1, True: 1, "False": 0, False: 0}).fillna(0).astype(int)

SEC = [(1,3,"A: Agriculture"),(5,9,"B: Mining"),(10,33,"C: Manufacturing"),(35,35,"D: Utilities"),
       (36,39,"E: Water/Waste"),(41,43,"F: Construction"),(45,47,"G: Retail/Wholesale"),
       (49,53,"H: Transport"),(55,56,"I: Accommodation/Food"),(58,63,"J: Information/Comms"),
       (64,66,"K: Finance/Insurance"),(68,68,"L: Real Estate"),(69,75,"M: Professional/Scientific"),
       (77,82,"N: Admin Support"),(84,84,"O: Public Admin"),(85,85,"P: Education"),
       (86,88,"Q: Health/Social"),(90,93,"R: Arts/Recreation"),(94,96,"S: Other Services"),
       (97,98,"T: Household Activities"),(99,99,"U: Extraterritorial")]

def section(code):
    """SIC code -> Companies House SIC section. 98 (property management) -> Real Estate."""
    try:
        div = int(str(code).strip()[:2])
    except (ValueError, TypeError):
        return None
    if div == 98:
        return "L: Real Estate"
    for lo, hi, s in SEC:
        if lo <= div <= hi:
            return s
    return None

d["sector"] = d["sic_code"].map(section)
print(f"sector mapped: {d['sector'].notna().mean():.1%}")

sector mapped: 100.0%


## 5 · Save

In [17]:
KEEP = ["com_num", "name", "sector", "account_type", "accounts_overdue",
        "nonlloyds_charges", "has_nonlloyds_charge", "yrs_since_nonlloyds_chg",
        "n_distinct_lenders", "age_years", "label"]

if "region" in d.columns and d["region"].notna().mean() > 0.5:
    # ~13.8k firms have no post_code at all, so no region can be derived -> label them
    # explicitly rather than leaving NaN, which would break sklearn downstream.
    d["region"] = d["region"].fillna("unknown")
    KEEP.insert(3, "region")
else:
    print("NOTE: 'region' missing/sparse -> excluded. Run Part 5 of GDELT.ipynb to add it.")

flat = d[KEEP].dropna(subset=["sector"]).reset_index(drop=True)
flat.to_csv(OUT_CSV, index=False)

print(f"saved {len(flat):,} rows | positives {int(flat['label'].sum()):,} "
      f"({flat['label'].mean():.2%})  ->  {OUT_CSV}")
flat.head(3)

NOTE: 'region' missing/sparse -> excluded. Run Part 5 of GDELT.ipynb to add it.
saved 311,991 rows | positives 10,122 (3.24%)  ->  /Users/natchalin_/Desktop/final_project/Lloyds/API/flat.csv


,com_num,name,sector,account_type,accounts_overdue,nonlloyds_charges,has_nonlloyds_charge,yrs_since_nonlloyds_chg,n_distinct_lenders,age_years,label
0,13172868,369 TECHNICAL LTD,C: Manufacturing,total-exemption-full,0,0,0,50.0,0,4.911704,0
1,07852962,369 UPLAND ROAD RTM COMPANY LIMITED,L: Real Estate,micro-entity,0,0,0,50.0,0,14.121834,0
2,14568066,369 VORTEX LIMITED,K: Finance/Insurance,micro-entity,0,0,0,50.0,0,2.995209,0


## 6 · Sanity checks

The first block is the one that matters: **no negative `yrs_since_nonlloyds_chg`**. A negative value
means a feature saw a charge filed after `ASOF` — the point-in-time bug this notebook fixes.

The age table is the exposure-time diagnostic. Under `"ever"` the oldest band has the highest rate
(they have had the most years to convert); under `"since"` that reverses, because every firm in the
window had equal opportunity. If you see a rising monotonic pattern, you are looking at exposure, not
propensity.

In [18]:
neg = int((flat["yrs_since_nonlloyds_chg"] < 0).sum())
print(f"negative yrs_since_nonlloyds_chg : {neg}   <- must be 0")
print(f"rows with any NaN                : {int(flat.isna().any(axis=1).sum())}")

print("\nmean by group (1 = customer, 0 = not):")
num = ["age_years", "nonlloyds_charges", "has_nonlloyds_charge",
       "yrs_since_nonlloyds_chg", "n_distinct_lenders"]
print(flat.groupby("label")[num].mean().round(2).to_string())

print("\nconversion rate by age band  (rising monotonic => exposure artefact):")
bands = pd.cut(flat["age_years"], [0, 5, 10, 15, 20, 30, 200],
               labels=["0-5", "5-10", "10-15", "15-20", "20-30", "30+"])
tab = flat.groupby(bands, observed=True)["label"].agg(["size", "mean"])
tab.columns = ["firms", "label_rate"]
print(tab.assign(label_rate=(tab["label_rate"] * 100).round(2)).to_string())

print("\ntop sectors by label rate (min 500 firms):")
s = flat.groupby("sector")["label"].agg(["size", "mean"])
s = s[s["size"] >= 500].sort_values("mean", ascending=False)
s.columns = ["firms", "label_rate"]
print(s.assign(label_rate=(s["label_rate"] * 100).round(2)).head(8).to_string())

negative yrs_since_nonlloyds_chg : 0   <- must be 0
rows with any NaN                : 0

mean by group (1 = customer, 0 = not):
       age_years  nonlloyds_charges  has_nonlloyds_charge  yrs_since_nonlloyds_chg  n_distinct_lenders
label                                                                                                 
0          10.45               0.48                  0.15                    43.72                0.27
1          25.15               3.01                  0.53                    30.08                1.38

conversion rate by age band  (rising monotonic => exposure artefact):
            firms  label_rate
age_years                    
0-5        105444        0.35
5-10        84645        1.24
10-15       49413        3.06
15-20       26329        5.34
20-30       30159       10.39
30+         15989       16.59

top sectors by label rate (min 500 firms):
                       firms  label_rate
sector                                  
A: Agriculture        

## 7 · Next — the lead list

`flat.csv` is the **training** table; it deliberately includes existing customers, because the model
learns from them. The RM lead list is the opposite slice: score every firm, then **drop
`label == 1`** and rank the remainder. Those high-scoring non-customers are the product (§9).

Feed this into the model notebook, and judge it with the same per-cutoff Precision@K + lift already
written there. Because each firm appears exactly once, there is no repeated-firm correlation — so no
cluster-bootstrap or `GroupKFold` is needed here.